In [1]:
# Fractual Sequence Model (Pre-trained on TinyStories)


## Architecture

The model is a symmetric U-Net stack of length-changing transformer layers:

```
embed -> [Compress]^N -> bottleneck -> [Decompress]^N -> head
              L -> L/K                             L*K <- L
```

- **Compress layer** = standard GPT block + `Conv1d(kernel=K, stride=K)`, mapping `L -> L/K`.
- **Decompress layer** = standard GPT block + `ConvTranspose1d(kernel=K, stride=K)`, mapping `L -> K*L`, then right-shifted by `K-1` for causality.
- **Skip connections** fuse same-resolution encoder features into the decoder.
- **Objective**: standard next-token cross-entropy.

In [2]:
import os, sys, math, time, random
import torch
import torch.nn.functional as F

sys.path.append(os.path.abspath("."))
from src.fsm import FractalSequenceModel, FSMConfig

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(0); random.seed(0)
print("device:", device)

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


device: mps


### 1. Tokenizer + TinyStories data

We use the GPT-2 BPE tokenizer (via `tiktoken`) and stream a slice of the
[`roneneldan/TinyStories`](https://huggingface.co/datasets/roneneldan/TinyStories)
dataset into a single contiguous token tensor for simple windowed sampling.

In [3]:
# pip install tiktoken datasets
import tiktoken
from datasets import load_dataset

enc = tiktoken.get_encoding("gpt2")
EOT = enc.eot_token  # 50256

# Pull a small slice for a quick prototype run; bump these for real training.
N_TRAIN_DOCS = 20000
N_VAL_DOCS   = 200

ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
train_ids, val_ids = [], []
for i, row in enumerate(ds):
    if i >= N_TRAIN_DOCS + N_VAL_DOCS: break
    toks = enc.encode_ordinary(row["text"]) + [EOT]
    (val_ids if i < N_VAL_DOCS else train_ids).extend(toks)

train_data = torch.tensor(train_ids, dtype=torch.long)
val_data   = torch.tensor(val_ids,   dtype=torch.long)
print(f"train tokens: {len(train_data):,}   val tokens: {len(val_data):,}")

README.md: 0.00B [00:00, ?B/s]

train tokens: 4,465,460   val tokens: 39,395


### 2. Build the Fractal Sequence Model

`block_size` must be divisible by `K ** n_levels`. With `K=4, n_levels=3` the
sequence is compressed `256 -> 64 -> 16 -> 4` tokens at the bottleneck, then
expanded back to 256.

In [4]:
cfg = FSMConfig(
    vocab_size = enc.n_vocab,    # 50257
    block_size = 256,
    n_levels   = 3,
    K          = 4,
    n_embd     = 256,
    n_head     = 8,
    mlp_ratio  = 4.0,
    dropout    = 0.0,
    use_skip   = True,
    mem_len    = 64,             # constant-length per-layer KV cache (0 = disabled)
)
model = FractalSequenceModel(cfg).to(device)
print(f"params: {model.num_params()/1e6:.2f} M")

# Effective receptive field at each scale: mem_len * K**s original tokens.
# (Plus the current chunk of block_size tokens at scale 0.)
for s in range(cfg.n_levels + 1):
    span = cfg.mem_len * (cfg.K ** s)
    label = "input" if s == 0 else (f"compress {s}" if s < cfg.n_levels else "bottleneck")
    print(f"  scale {s} ({label}): cache covers {span:>6d} original tokens "
          f"({'verbatim' if s == 0 else 'compressed x' + str(cfg.K**s)})")

# Quick shape sanity-check (stateless mode -> back-compat 2-tuple)
with torch.no_grad():
    x = torch.randint(0, cfg.vocab_size, (2, cfg.block_size), device=device)
    logits, loss = model(x, x)
    print("logits:", tuple(logits.shape), "  loss:", float(loss))

params: 20.37 M
  scale 0 (input): cache covers     64 original tokens (verbatim)
  scale 1 (compress 1): cache covers    256 original tokens (compressed x4)
  scale 2 (compress 2): cache covers   1024 original tokens (compressed x16)
  scale 3 (bottleneck): cache covers   4096 original tokens (compressed x64)
logits: (2, 256, 50257)   loss: 10.867584228515625


### 3. Training loop

In [5]:
def get_batch(split: str, batch_size: int):
    data = train_data if split == "train" else val_data
    T = cfg.block_size
    ix = torch.randint(0, data.size(0) - T - 1, (batch_size,))
    x = torch.stack([data[i     : i + T    ] for i in ix])
    y = torch.stack([data[i + 1 : i + T + 1] for i in ix])
    return x.to(device, non_blocking=True), y.to(device, non_blocking=True)

@torch.no_grad()
def estimate_loss(batch_size=8, n_iters=20):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(n_iters)
        for i in range(n_iters):
            x, y = get_batch(split, batch_size)
            _, loss = model(x, y)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Hyperparameters (tune up if you have a GPU)
BATCH_SIZE   = 16
MAX_ITERS    = 1000
EVAL_EVERY   = 100
LR           = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP    = 1.0

optim = torch.optim.AdamW(model.parameters(), lr=LR,
                          betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)

model.train()
t0 = time.time()
for it in range(1, MAX_ITERS + 1):
    x, y = get_batch("train", BATCH_SIZE)
    _, loss = model(x, y)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optim.step()

    if it % EVAL_EVERY == 0 or it == 1:
        m = estimate_loss()
        dt = time.time() - t0
        print(f"iter {it:5d} | train {m['train']:.3f} | val {m['val']:.3f} | {dt:.1f}s")

iter     1 | train 10.889 | val 10.876 | 3.6s
iter   100 | train 6.032 | val 6.150 | 18.2s
iter   200 | train 5.991 | val 6.114 | 35.3s
iter   300 | train 5.989 | val 6.090 | 53.1s
iter   400 | train 5.870 | val 5.994 | 71.6s
iter   500 | train 5.614 | val 5.751 | 90.8s
iter   600 | train 5.375 | val 5.542 | 114.6s
iter   700 | train 5.272 | val 5.416 | 158.6s
iter   800 | train 5.113 | val 5.324 | 207.0s
iter   900 | train 5.011 | val 5.181 | 298.1s
iter  1000 | train 4.882 | val 5.081 | 365.2s


### 4. Sample from the model

In [6]:
prompt = "Once upon a time"
ids = torch.tensor([enc.encode_ordinary(prompt)], dtype=torch.long, device=device)
out = model.generate(ids, max_new_tokens=120, temperature=0.9, top_k=50)
print(enc.decode(out[0].tolist()))

Once upon a time. Let. They did to the friends with the toys!" Lily. I you you to the ball. She was to playing. The little friend, very careful on a it said. She to the Lily the play in something. "That and to not?" He� Lily!" Spot to to the fun was you. " girl said and to it and you to the go. The day!

Tom was they, Lily was the house to. Maybe not a bird.

When looked, a girl came. He, the time were you.

"I the a big saw


In [8]:
model

FractalSequenceModel(
  (tok_emb): Embedding(50257, 256)
  (drop): Dropout(p=0.0, inplace=False)
  (compress): ModuleList(
    (0-2): 3 x CompressLayer(
      (block): GPTBlock(
        (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (attn): CausalSelfAttention(
          (qkv): Linear(in_features=256, out_features=768, bias=True)
          (proj): Linear(in_features=256, out_features=256, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=256, out_features=1024, bias=True)
          (fc2): Linear(in_features=1024, out_features=256, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
      (ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (conv): Conv1d(256, 256, kernel_size=(4,), stride=(4,))
    )
  )
  (bottleneck): GPTBlock

### 5. Memory-augmented (chunked) training

We feed the model a stream of contiguous chunks and pass an `FSMState`
across calls. Each layer keeps a **constant-length** KV cache (`cfg.mem_len`),
so memory cost per layer is fixed but the *receptive field* grows
geometrically with depth: layer `s` sees `mem_len · K^s` past tokens.

Cache entries are detached at chunk boundaries (Transformer-XL style):
gradients flow within a chunk, the cache acts as read-only context.

In [7]:
def get_stream_batch(split: str, batch_size: int, n_chunks: int):
    """Sample `batch_size` long contiguous windows of `n_chunks * block_size + 1`
    tokens, then split into chunk-aligned (x, y) pairs."""
    data = train_data if split == "train" else val_data
    T = cfg.block_size
    span = n_chunks * T
    ix = torch.randint(0, data.size(0) - span - 1, (batch_size,))
    seq = torch.stack([data[i : i + span + 1] for i in ix])    # (B, span+1)
    xs = [seq[:, c * T     : (c + 1) * T    ] for c in range(n_chunks)]
    ys = [seq[:, c * T + 1 : (c + 1) * T + 1] for c in range(n_chunks)]
    return xs, ys

# Hyperparams for the memory-mode demo
N_CHUNKS    = 4               # effective context = N_CHUNKS * block_size = 1024 tokens
BATCH_SIZE_M = 8
MAX_ITERS_M = 300
EVAL_EVERY_M = 50
LR_M        = 3e-4

optim = torch.optim.AdamW(model.parameters(), lr=LR_M,
                          betas=(0.9, 0.95), weight_decay=0.1)

@torch.no_grad()
def estimate_loss_stream(batch_size=4, n_iters=10):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(n_iters):
            xs, ys = get_stream_batch(split, batch_size, N_CHUNKS)
            state = model.init_state()
            for x, y in zip(xs, ys):
                _, loss, state = model(x.to(device), y.to(device), state=state)
                losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

model.train()
t0 = time.time()
for it in range(1, MAX_ITERS_M + 1):
    xs, ys = get_stream_batch("train", BATCH_SIZE_M, N_CHUNKS)
    state = model.init_state()
    optim.zero_grad(set_to_none=True)
    chunk_losses = []
    for x, y in zip(xs, ys):
        # state holds detached KVs from previous chunk -> no graph build-up
        _, loss, state = model(x.to(device), y.to(device), state=state)
        loss.backward()                          # within-chunk BPTT only
        chunk_losses.append(loss.item())
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optim.step()

    if it % EVAL_EVERY_M == 0 or it == 1:
        m = estimate_loss_stream()
        dt = time.time() - t0
        print(f"iter {it:4d} | last-chunk train {chunk_losses[-1]:.3f} "
              f"| eval train {m['train']:.3f} val {m['val']:.3f} | {dt:.1f}s")

iter    1 | last-chunk train 7.031 | eval train 6.664 val 6.774 | 8.1s
iter   50 | last-chunk train 5.004 | eval train 4.981 val 5.231 | 60.8s
iter  100 | last-chunk train 4.957 | eval train 4.873 val 4.988 | 107.5s
iter  150 | last-chunk train 4.608 | eval train 4.840 val 5.035 | 150.0s
iter  200 | last-chunk train 4.633 | eval train 4.668 val 4.914 | 192.1s
iter  250 | last-chunk train 4.841 | eval train 4.614 val 4.819 | 233.0s
iter  300 | last-chunk train 4.600 | eval train 4.538 val 4.693 | 271.2s


In [ ]:
# The fractual memory is beautiful, I want to explore how to use it to beat 
# standard GPT in long context, under the SAME vram budget. 